# EDA: Прогноз оттока клиентов телекоммуникационной компании

## Бизнес-контекст

Телекоммуникационная компания теряет клиентов (отток, *churn*), что напрямую снижает выручку.
Удержать существующего клиента, как правило, дешевле, чем привлечь нового, поэтому отделу
маркетинга важно **заранее** знать, какие клиенты находятся в зоне риска, чтобы вовремя
запустить персональную удерживающую кампанию (скидка, апгрейд тарифа, звонок менеджера).

## Датасет

Используется датасет **IBM Telco Customer Churn**:
- 7043 строки (клиенты), 21 признак (услуги, тип контракта, платёжная история, демография)
- Целевая переменная — `Churn` (`Yes`/`No`): покинул ли клиент компанию

## Цель анализа

1. Понять качество и структуру данных (пропуски, типы, распределения).
2. Найти признаки, наиболее сильно связанные с оттоком.
3. Сформировать гипотезы, которые будут проверены и использованы на этапе моделирования
   (`02_modeling.ipynb`).


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
sns.set_palette("Set2")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

pd.set_option("display.max_columns", None)


In [ ]:
DATA_URL_PRIMARY = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)
DATA_URL_FALLBACK = (
    "https://raw.githubusercontent.com/dsrscientist/dataset1/master/telecom_churn.csv"
)

try:
    df = pd.read_csv(DATA_URL_PRIMARY)
except Exception as e:
    print(f"Основной источник недоступен ({e}), пробуем резервный URL...")
    df = pd.read_csv(DATA_URL_FALLBACK)

print("Размер датасета:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe(include="all").T


In [ ]:
print("Пропущенные значения по столбцам:")
df.isnull().sum()


## Вывод по качеству данных

- Явных пропусков (`NaN`), определяемых `isnull()`, в датасете практически нет — однако
  столбец `TotalCharges` загружен как **строковый (object)** тип, хотя по смыслу это число.
  Это значит, что часть значений (как правило, у клиентов с `tenure == 0`, то есть новых
  клиентов, ещё не оплативших ни одного месяца) содержит пробелы вместо чисел и будет
  потеряна при прямом приведении к `float` без дополнительной обработки.
- Большинство признаков — категориальные (`object`) с небольшим числом уникальных значений
  (`Yes`/`No`, тип контракта, способ оплаты и т.д.) — это типичный кандидат для
  `OneHotEncoder` на этапе моделирования.
- Числовые признаки: `tenure` (срок обслуживания в месяцах), `MonthlyCharges`
  (ежемесячный платёж), `SeniorCitizen` (бинарный флаг, хотя хранится как `int`).
- Перед моделированием потребуется: привести `TotalCharges` к `float` (с `errors="coerce"` +
  заполнением пропусков медианой) и закодировать целевую переменную `Churn` в `0`/`1`.


In [ ]:
churn_counts = df["Churn"].value_counts()
churn_rate = (churn_counts["Yes"] / len(df)) * 100

plt.figure(figsize=(6, 5))
ax = sns.countplot(data=df, x="Churn", order=["No", "Yes"])
ax.set_title("Распределение целевой переменной Churn")
ax.set_xlabel("Отток клиента")
ax.set_ylabel("Количество клиентов")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()

print(f"Доля ушедших клиентов (Churn rate): {churn_rate:.1f}%")


## Вывод по балансу классов

Наблюдается умеренный **дисбаланс классов**: около 73% клиентов остаются, и около 27%
уходят (соотношение примерно 2.7:1). Это не критичный, но значимый перекос.

**Следствие для выбора метрики:** при таком дисбалансе **accuracy** — плохой выбор метрики,
так как модель, всегда предсказывающая "клиент остаётся", уже получит ~73% точности, не
выявив ни одного реального оттока. Поэтому на этапе моделирования приоритет отдаётся
**ROC-AUC** (устойчива к дисбалансу, оценивает ранжирование) в комбинации с **F1**,
**Precision**, **Recall** и **PR-AUC**, которые показывают, насколько хорошо модель находит
именно класс "уйдёт" — а это и есть бизнес-ценность сервиса.


In [ ]:
contract_churn = (
    df.groupby("Contract")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .sort_values(ascending=False)
)

plt.figure(figsize=(7, 5))
ax = sns.barplot(x=contract_churn.index, y=contract_churn.values, order=contract_churn.index)
ax.set_title("Churn rate по типу контракта")
ax.set_xlabel("Тип контракта")
ax.set_ylabel("Churn rate, %")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()

contract_churn


## Вывод: тип контракта и отток

Клиенты с помесячным контрактом (**Month-to-month**) уходят значительно чаще, чем клиенты
с годовым или двухлетним контрактом. Это полностью согласуется с бизнес-логикой: помесячный
контракт не накладывает обязательств и не даёт скидки за долгосрочность, поэтому клиенту
психологически и финансово легче расторгнуть его в любой момент. Долгосрочные контракты
(One year, Two year), как правило, включают более выгодные условия и "привязывают" клиента
надолго, снижая вероятность ухода. Тип контракта — один из главных кандидатов в важные
признаки для модели.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", order=["No", "Yes"], ax=axes[0])
axes[0].set_title("MonthlyCharges: boxplot по Churn")

sns.kdeplot(data=df[df["Churn"] == "No"], x="MonthlyCharges", label="Churn = No", fill=True, ax=axes[1])
sns.kdeplot(data=df[df["Churn"] == "Yes"], x="MonthlyCharges", label="Churn = Yes", fill=True, ax=axes[1])
axes[1].set_title("MonthlyCharges: плотность распределения по Churn")
axes[1].legend()

plt.tight_layout()
plt.show()


## Вывод: ежемесячный платёж и отток

Ушедшие клиенты (`Churn = Yes`) в среднем платят **больше**, чем оставшиеся: медиана
`MonthlyCharges` у ушедших заметно выше, а плотность распределения для `Churn = Yes` смещена
в область более высоких сумм (особенно заметен пик в диапазоне высоких платежей за
Fiber optic интернет). Это говорит о том, что более высокая ежемесячная стоимость услуг
повышает чувствительность клиента к цене и снижает воспринимаемую выгоду — клиент охотнее
ищет альтернативы или просит отключение, если ощущает, что платит слишком много.


In [ ]:
plt.figure(figsize=(8, 5))
sns.kdeplot(data=df[df["Churn"] == "No"], x="tenure", label="Churn = No", fill=True)
sns.kdeplot(data=df[df["Churn"] == "Yes"], x="tenure", label="Churn = Yes", fill=True)
plt.title("Распределение tenure (срок обслуживания, мес.) по Churn")
plt.xlabel("tenure, месяцев")
plt.legend()
plt.tight_layout()
plt.show()


## Вывод: срок обслуживания и отток

Распределение `tenure` для ушедших клиентов сильно смещено влево — основная масса оттока
происходит в **первые месяцы** обслуживания. Клиенты, проработавшие с компанией много лет,
уходят значительно реже. Интерпретация: "молодые" клиенты ещё не успели оценить полную
выгоду от услуг, не привязаны накопленными бонусами/привычками и более чувствительны к
первому негативному опыту (проблемы с качеством связи, биллингом, поддержкой). Это говорит
о важности программ адаптации и контроля качества именно в первые 3–6 месяцев
обслуживания.


In [ ]:
internet_churn = (
    df.groupby("InternetService")["Churn"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .sort_values(ascending=False)
)

plt.figure(figsize=(7, 5))
ax = sns.barplot(x=internet_churn.index, y=internet_churn.values, order=internet_churn.index)
ax.set_title("Churn rate по типу интернет-услуги")
ax.set_xlabel("InternetService")
ax.set_ylabel("Churn rate, %")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()

internet_churn


## Вывод: тип интернет-услуги и отток

Клиенты с подключением **Fiber optic** уходят значительно чаще, чем клиенты с DSL или без
интернета вообще. На первый взгляд это нелогично — оптика обычно качественнее DSL, — но это
согласуется с предыдущим выводом о `MonthlyCharges`: тариф на Fiber optic заметно дороже,
а более высокая цена повышает требовательность клиента и риск ухода при любом недовольстве
качеством или сервисом. Клиенты без интернета (`No`) — самая стабильная группа, вероятно,
из-за минимального набора услуг и низких счетов.


In [ ]:
df_corr = df.copy()
df_corr["TotalCharges"] = pd.to_numeric(df_corr["TotalCharges"], errors="coerce")
df_corr["TotalCharges"] = df_corr["TotalCharges"].fillna(df_corr["TotalCharges"].median())
df_corr["Churn_num"] = (df_corr["Churn"] == "Yes").astype(int)

numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "Churn_num"]
corr_matrix = df_corr[numeric_cols].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Корреляции числовых признаков и Churn")
plt.tight_layout()
plt.show()


## Итоговые выводы EDA

1. **Дисбаланс классов** — отток составляет около 27% клиентов; для оценки моделей нужно
   использовать ROC-AUC, F1, Precision, Recall, PR-AUC, а не accuracy.
2. **Тип контракта** — клиенты на помесячном контракте (`Month-to-month`) уходят значительно
   чаще, чем клиенты с годовыми/двухлетними контрактами. Сильный кандидат в топ-признаки.
3. **Ежемесячный платёж** — ушедшие клиенты в среднем платят больше; высокая стоимость услуг
   коррелирует с повышенным риском оттока.
4. **Срок обслуживания (tenure)** — основная доля оттока приходится на первые месяцы
   сотрудничества с компанией; "стаж" клиента — сильный защитный фактор.
5. **Тип интернет-услуги** — клиенты на Fiber optic уходят чаще DSL и клиентов без интернета,
   что коррелирует с более высокой стоимостью этого тарифа.

**Гипотезы для этапа моделирования:**
- Комбинация `Contract` + `tenure` + `MonthlyCharges` + `InternetService`, вероятно, даст
  наибольший вклад в предсказательную силу модели.
- Корреляция числовых признаков с `Churn_num` умеренная и нелинейная — это говорит в пользу
  древовидных моделей (Decision Tree, LightGBM), способных находить нелинейные
  взаимодействия признаков, по сравнению с чисто линейной Logistic Regression.
- Перед обучением необходимо корректно привести `TotalCharges` к числовому типу и
  обработать пропуски, возникающие у новых клиентов (`tenure == 0`).
